In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if temperature is not None:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [3]:
# Send a PDF as a base64 document block placed before the text prompt.
# Limits: 32 MB per request and up to 600 pages
with open("earth.pdf", "rb") as f:
    pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": pdf_data,
            },
        },
        {"type": "text", "text": "Summarize the document in one sentence."},
    ],
)

response = chat(messages)
print(text_from_message(response))

This Wikipedia article on Earth describes it as the third planet from the Sun and the only known body to harbor life, detailing its physical characteristics (size, mass, composition), orbital and rotational dynamics, atmosphere, water systems, geology, magnetosphere, the etymology of its name and related adjectives, and its natural history from formation roughly 4.5 billion years ago through the development of life, human emergence, and humanity's growing environmental impact.


In [4]:
# Enable citations on the document — the answer comes back as text blocks that
# carry page_location citations pointing at the supporting PDF pages
messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": pdf_data,
            },
            "title": "Earth.pdf",
            "citations": {"enabled": True},
        },
        {"type": "text", "text": "How were Earth's atmosphere and oceans formed?"},
    ],
)

chat(messages)

Message(id='msg_011Cd6jq1oYhqJgew3AdxFkc', container=None, content=[TextBlock(citations=None, text="Based on the document, Earth's atmosphere and oceans have a volcanic and cosmic origin:\n\n**Formation of the atmosphere and oceans:**\n", type='text'), TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n", document_index=0, document_title='Earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing. Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.", type='text'), TextBlock(citations=None, text='\n\n**Water may have been present from the start:**\nInterestingly, ', type